# CBMLoss: Concept Bottleneck Models com Mitigação de Concept Leakage
Este notebook está configurado para rodar diretamente dentro da pasta do projeto montada a partir do seu Google Drive (`G:\Meu Drive\Projetos\Python\CBMLoss`).

**Vantagens:**
- Não é necessário clonar nada via Git;
- O dataset CUB-200 já existente no seu Google Drive é utilizado diretamente;
- Todos os novos checkpoints, CSVs e figuras gerados são salvos diretamente no seu Drive e sincronizados para sua máquina local.

In [ ]:
# =============================================================================
# 1. CONECTAR AO GOOGLE DRIVE E NAVEGAR PARA A PASTA DO PROJETO
# =============================================================================
import os
from google.colab import drive

# 1. Monta o Google Drive
drive.mount('/content/drive')

# 2. Localiza automaticamente a pasta do projeto CBMLoss dentro do Google Drive
candidatos = [
    '/content/drive/MyDrive/Projetos/Python/CBMLoss',
    '/content/drive/My Drive/Projetos/Python/CBMLoss',
    '/content/drive/Meu Drive/Projetos/Python/CBMLoss',
    '/content/drive/Shareddrives/Projetos/Python/CBMLoss'
]

caminho_projeto = None
for c in candidatos:
    if os.path.isdir(c) and os.path.exists(os.path.join(c, 'main.py')):
        caminho_projeto = c
        break

if caminho_projeto is None:
    print('Buscando pasta CBMLoss dentro do Google Drive...')
    for root, dirs, _ in os.walk('/content/drive'):
        if 'CBMLoss' in dirs:
            c = os.path.join(root, 'CBMLoss')
            if os.path.exists(os.path.join(c, 'main.py')):
                caminho_projeto = c
                break

if caminho_projeto:
    os.chdir(caminho_projeto)
    get_ipython().run_line_magic('cd', caminho_projeto)
    print(f'>>> Sucesso! Navegando diretamente em: {os.getcwd()}')
    if os.path.exists('data/CUB_200_2011/train.csv'):
        print('>>> Dataset CUB-200 detectado com sucesso em data/CUB_200_2011!')
        print(f'>>> Total de arquivos em data/CUB_200_2011: {len(os.listdir("data/CUB_200_2011"))}')
    else:
        print('>>> ATENÇÃO: Dataset ainda não detectado em data/CUB_200_2011.')
else:
    raise FileNotFoundError('Pasta do projeto CBMLoss não encontrada no Google Drive!')


In [ ]:
# =============================================================================
# 2. INSTALAÇÃO DE DEPENDÊNCIAS NO AMBIENTE COLAB
# =============================================================================
import os
if not os.path.exists('requirements.txt'):
    for c in ['/content/drive/MyDrive/Projetos/Python/CBMLoss', '/content/drive/My Drive/Projetos/Python/CBMLoss', '/content/drive/Meu Drive/Projetos/Python/CBMLoss']:
        if os.path.isdir(c) and os.path.exists(os.path.join(c, 'main.py')):
            os.chdir(c)
            get_ipython().run_line_magic('cd', c)
            break
!pip install -r requirements.txt -q
print('>>> Dependências instaladas com sucesso!')


## FASE 2: Estudos de Ablação Desacoplada e Medição Direta de Leakage
Os experimentos abaixo respondem diretamente aos pedidos dos revisores do SIBGRAPI:
1. **Baseline sem Regularização:** $\lambda_{ent}=0.0, \lambda_{ortho}=0.0$
2. **Isolamento da Entropia:** $\lambda_{ent}=0.5, \lambda_{ortho}=0.0$
3. **Isolamento da Descorrelação:** $\lambda_{ent}=0.0, \lambda_{ortho}=0.5$
4. **Configurações Leves Isoladas:** $0.1 / 0.0$ e $0.0 / 0.1$
5. **Medição Direta de Leakage:** Gap contínuo-discreto e Sonda Linear sobre o ruído residual dos conceitos.

In [ ]:
# Baseline: Modelo CBM sem regularização (lambda_ent=0.0, lambda_ortho=0.0)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Ablação 1: Apenas Entropia (lambda_ent=0.5, lambda_ortho=0.0)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.5 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Ablação 2: Apenas Descorrelação (lambda_ent=0.0, lambda_ortho=0.5)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.5 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Ablação 3 (Leve): Apenas Entropia (lambda_ent=0.1, lambda_ortho=0.0)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.1 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Ablação 4 (Leve): Apenas Descorrelação (lambda_ent=0.0, lambda_ortho=0.1)
!python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.1 --pretrained --checkpoint_dir checkpoints --patience 5


### Medição Direta de Concept Leakage
Executa o script de medição direta (Linear Probing sobre resíduos e gap de discretização) em todos os modelos salvos em `checkpoints/` e gera o arquivo `leakage_direct_metrics.csv` diretamente no projeto.

In [ ]:
# Medição Direta de Leakage em todos os checkpoints disponíveis
!python measure_leakage.py --dataset cub200 --checkpoint_dir checkpoints --output_csv leakage_direct_metrics.csv


## Histórico: Execuções Anteriores (Ablação Conjunta $\lambda_{ent} = \lambda_{ortho}$)
Células abaixo mantidas como referência dos modelos já treinados anteriormente.

In [ ]:
# Treino original: lambda=0.0
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.0 --lambda_ortho 0.0 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Treino original: lambda=0.1
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.1 --lambda_ortho 0.1 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Treino original: lambda=0.3
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.3 --lambda_ortho 0.3 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Treino original: lambda=0.5
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.5 --lambda_ortho 0.5 --pretrained --checkpoint_dir checkpoints --patience 5


In [ ]:
# Treino original: lambda=0.7
# !python main.py --dataset cub200 --epochs 100 --lambda_ent 0.7 --lambda_ortho 0.7 --pretrained --checkpoint_dir checkpoints --patience 5
